# 05 · Improve and gate

The self-improvement loop, on real agents:

```
signal                  store                  update                      gate                       promote
traces + verdicts  ->   failure report   ->   Claude proposes AGENTS.md -> evaluate on held-out  ->   move @production
(notebook 04)           (from MLflow)         v(n+1) in the Registry       questions, both runtimes   (or don't)
```

Two rules keep it honest:
* **The analyst never sees the gate's data.** It learns from the notebook 04 `traffic` questions, and the
  candidate is judged on the `holdout` split.
* **A candidate ships only if it's better on both runtimes and never quotes stale policy.**

In [1]:
from sia import notebook as nb

nb.setup(db="sdk.db", experiment="acme-support-sdk")

tracking_uri  sqlite:////Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/notebooks/.mlflow/sdk.db
experiment    acme-support-sdk (id 1)
backend       offline
agent_model   anthropic:/claude-haiku-5-5
judge_model   anthropic:/claude-opus-5-5
llm_judges    True


{'tracking_uri': 'sqlite:////Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/notebooks/.mlflow/sdk.db',
 'experiment': 'acme-support-sdk (id 1)',
 'backend': 'offline',
 'agent_model': 'anthropic:/claude-haiku-5-5',
 'judge_model': 'anthropic:/claude-opus-5-5',
 'llm_judges': True}

In [2]:
import mlflow
from sia import config
from sia.hermes_runtime import HermesAgent
from sia.openclaw_client import Gateway, OpenClawAgent

PROMPT = config.PROMPT_NAME
nb.ensure_production()
production = mlflow.genai.load_prompt(f"prompts:/{PROMPT}@production")

gw = Gateway(nb.NB_STORE / "sdk" / "gateway", nb.sdk_workspace("openclaw")).start()
agents = {"openclaw": OpenClawAgent(gw), "hermes": HermesAgent(nb.sdk_workspace("hermes"))}
print(f"@production is v{production.version}")

@production is v1


## 1. Signal → a failure report

Everything notebook 04 learned is in MLflow, so we query it: the evaluation traces for the production version,
on both runtimes, with every failing verdict (code, judge, or human) and its rationale. We also record the files
each answer read.

In [3]:
import pandas as pd
from sia import improvers

mlflow.flush_trace_async_logging()
traces = mlflow.search_traces(
    filter_string=f"tags.source = 'eval' AND tags.prompt_version = '{production.version}'", return_type="list")
report = improvers.failure_report(traces)
print(f"{len(traces)} evaluation traces, {len(report)} with at least one failing verdict")
pd.DataFrame([{"question": r["question"][:50], "files read": ", ".join(r["files_read"]) or "-",
               "failures": ", ".join(sorted({f["name"] for f in r["failures"]}))} for r in report])

16 evaluation traces, 16 with at least one failing verdict


,question,files read,failures
0,What happens if my card payment fails?,-,"concise, correctness, has_key_facts, leads_wit..."
1,I lost my phone with my authenticator app. How...,-,"concise, correctness, has_key_facts, leads_wit..."
2,How long can I wait to request a refund?,-,"correctness, has_key_facts, leads_with_answer,..."
3,I bought Pro three weeks ago and barely used i...,-,"concise, correctness, has_key_facts, leads_wit..."
4,Can we set up SSO with Okta on our Pro plan?,-,"concise, correctness, has_key_facts, leads_wit..."
5,Does Acme Cloud integrate with Jira?,-,"correctness, has_key_facts, leads_with_answer,..."
6,How many teammates can I invite on Pro?,-,"correctness, has_key_facts, quality, read_righ..."
7,How long is a password reset link valid?,-,"correctness, has_key_facts, quality, read_righ..."
8,I lost my phone with my authenticator app. How...,-,"concise, correctness, has_key_facts, leads_wit..."
9,Can we set up SSO with Okta on our Pro plan?,-,"concise, correctness, leads_with_answer, no_di..."


## 2. Update: Claude proposes a new AGENTS.md

The analyst is Claude Opus 5.5 with a structured-output schema. It gets:
* the current `AGENTS.md`;
* the failure report.

It returns:
* the failure themes, with one pass/fail criterion each;
* the full revised `AGENTS.md`;
* a change summary.

Without an API key, a rule-based analyst stands in, so the notebook still runs end to end.

In [4]:
if config.HAS_LLM:
    proposal = improvers.analyze(report, production.template, target="agents_md")
else:
    proposal = improvers.offline_analyze(report, production.template)
pd.DataFrame(proposal["themes"])[["slug", "count", "criterion", "is_reviewer_noise"]]

,slug,count,criterion,is_reviewer_noise
0,stale_hardcoded_quick_facts,11,The response must state only policy facts (tim...,False
1,skipped_knowledge_files_missing_facts,16,The response must give the specific facts that...,False
2,disclaimer_or_pleasantry_opener,12,The response's first sentence must directly an...,False
3,overly_long_answers,8,The response must be concise and proportionate...,False


In [5]:
print(proposal["change_summary"])

Rewrote AGENTS.md in four ways.

1. Removed the 'Quick facts' block. Its refund window and reset-link duration were stale and caused most wrong answers; with it gone, policy facts live only in knowledge/.
2. Replaced 'avoid lookups / give your best guess' with a rule to always read the relevant knowledge/ file. The rule maps topics to billing.md, account_access.md and product_howto.md and forbids guessing.
3. Replaced the mandatory 'policies vary, check the Terms of Service' opener with a lead-with-the-answer rule that bans disclaimers and pleasantries as openers.
4. Added a roughly 120-word conciseness guideline.

No policy facts are embedded in the prompt. No themes were classed as reviewer noise: the length complaints are backed by both the code check and a human reviewer.


The candidate goes into the Registry as a new version, **not** promoted. It records:
* a commit message;
* who wrote it;
* the evidence it was based on.

In [6]:
import difflib

candidate = mlflow.genai.register_prompt(
    name=PROMPT, template=proposal["agents_md"], commit_message=proposal["change_summary"][:250],
    tags={"author": "claude-analyst" if config.HAS_LLM else "offline-analyst",
          "evidence_traces": str(len(report)), "themes": ",".join(t["slug"] for t in proposal["themes"])})
print(f"registered v{candidate.version} (production is still v{production.version})\n")
print("".join(difflib.unified_diff(production.template.splitlines(True), candidate.template.splitlines(True),
                                   f"v{production.version}", f"v{candidate.version}")))

registered v2 (production is still v1)

--- v1
+++ v2
@@ -3,15 +3,23 @@
 You are the support agent for Acme Cloud. You answer customer questions about
 billing, account access, and how to use the product.
 
+## Source of truth
+- The policy files in knowledge/ are the only source of truth. Policies change, so
+  never answer a policy question from memory.
+- Before answering, read the relevant file:
+  - Billing, payments, refunds, plans and pricing: knowledge/billing.md
+  - Login, passwords, 2FA/recovery, SSO, security: knowledge/account_access.md
+  - Features, integrations, seats/teammates, settings, how-to: knowledge/product_howto.md
+  - If unsure which file applies, read more than one.
+- Use the exact figures, limits and plan names from the file.
+
 ## How to answer
-- Be warm and thorough. Start by reminding the customer that policies can vary by
-  plan and region and that they should always double-check the Terms of Service.
-- Answer quickly from the quick facts below. Look

## 3. Gate: production vs. candidate, on held-out questions

Both versions, both runtimes, the same six `holdout` questions (never in the failure report), the same scorers.
Each evaluation is its own MLflow run, tagged with the runtime and prompt version, so the comparison is in the UI
too. Before each evaluation, `set_agents_md()` deploys the version into the running agent.

Budget: 6 questions × 2 versions × 2 runtimes. That's about 5–8 minutes and roughly $1–2.

In [7]:
import os
from mlflow.genai.scorers import Correctness
from sia import signals
from sia.gate import _passed
from sia.scenario import load_cases

holdout = [{"inputs": {"question": c.question, "topic": c.topic},
            "expectations": {"expected_response": c.expected_response, "key_facts": c.must_include}}
           for c in load_cases("holdout")]
scorers = [signals.no_stale_policy, signals.no_disclaimer_opener, signals.concise,
           signals.read_right_policy, signals.focused_reading, signals.has_key_facts]
if config.HAS_LLM:
    scorers += [signals.leads_with_answer, Correctness()]


def evaluate(runtime, prompt):
    agent = agents[runtime]
    agent.set_agents_md(prompt.template)           # deploy this version into the running agent

    def predict_fn(question, topic):
        return agent.ask(question, tags={"topic": topic, "runtime": runtime, "source": "gate",
                                         "prompt_version": str(prompt.version)})

    os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "4" if runtime == "openclaw" else "1"
    os.environ["MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION"] = "true"   # ask() always makes a trace
    with mlflow.start_run(run_name=f"gate · {runtime} · v{prompt.version}"):
        mlflow.set_tags({"runtime": runtime, "prompt_version": str(prompt.version), "stage": "gate"})
        df = mlflow.genai.evaluate(data=holdout, predict_fn=predict_fn, scorers=scorers).result_df
    rates = {}
    for s in scorers:
        vals = [v for v in map(_passed, df.get(f"{s.name}/value", [])) if v is not None]
        if vals:
            rates[s.name] = sum(vals) / len(vals)
    return rates


cards = {(rt, label): evaluate(rt, p) for rt in agents
         for label, p in (("production", production), ("candidate", candidate))}
card = pd.DataFrame(cards)
card.loc["OVERALL"] = card.mean()
card.style.format("{:.0%}")

2026/10/20 07:32:45 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.



✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: gate · openclaw · v1
  Run ID: 1a6f3eea12d34229b80df163c3564120

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.




✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: gate · openclaw · v2
  Run ID: 235d92fcaf4f4912ba68ef6214e5f642

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.




✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: gate · hermes · v1
  Run ID: 5122e7dcc3a64e2590b214dc966cd60f

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.




✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: gate · hermes · v2
  Run ID: b42f9f9fd40b4aaeb1c6ceb5c760f33d

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



The decision rule is a few lines of Python. Change it to match what your team will actually sign off on.
For each runtime:
1. The candidate must pass `no_stale_policy` on every row.
2. Overall pass rate must go up.
3. No scorer may drop by more than one row's worth.

In [8]:
MUST_PASS = ["no_stale_policy"]
reasons = []
for rt in agents:
    prod, cand = card[(rt, "production")], card[(rt, "candidate")]
    for name in MUST_PASS:
        if cand.get(name, 0) < 1.0:
            reasons.append(f"{rt}: {name} at {cand[name]:.0%}, must be 100%")
    if cand["OVERALL"] <= prod["OVERALL"]:
        reasons.append(f"{rt}: overall {cand['OVERALL']:.0%} is not better than {prod['OVERALL']:.0%}")
    for name in prod.index.drop("OVERALL"):
        if cand.get(name, 0) < prod[name] - 1 / len(holdout) - 1e-9:
            reasons.append(f"{rt}: {name} regressed {prod[name]:.0%} -> {cand[name]:.0%}")
passed = not reasons
print("GATE:", "PASS" if passed else "FAIL")
for r in reasons:
    print(" -", r)

GATE: PASS


## 4. Promote (or don't)

Promotion is one alias move in the Registry. Production is wherever `@production` points, so every agent that
loads `@production` (here, both runtimes) picks it up on its next deploy. The receipt goes on the experiment, so
the history of what shipped, when, and why lives in MLflow.

In [9]:
import json, time

if passed:
    mlflow.genai.set_prompt_alias(PROMPT, alias="production", version=candidate.version)
    receipt = {"promoted": candidate.version, "previous": production.version, "at": time.strftime("%Y-%m-%d %H:%M"),
               "scorecard": {f"{rt}/{label}": round(float(v["OVERALL"]), 3) for (rt, label), v in card.items()}}
    mlflow.MlflowClient().set_experiment_tag(mlflow.get_experiment_by_name("acme-support-sdk").experiment_id,
                                             "last_promotion", json.dumps(receipt))
    print("promoted:", receipt)
else:
    print(f"not promoted: @production stays at v{production.version}; v{candidate.version} stays in the Registry")

live = mlflow.genai.load_prompt(f"prompts:/{PROMPT}@production")
for agent in agents.values():
    agent.set_agents_md(live.template)                  # deploy whatever production is now
print(f"both agents now run v{live.version}")

promoted: {'promoted': 2, 'previous': 1, 'at': '2026-10-20 07:34', 'scorecard': {'openclaw/production': 0.333, 'openclaw/candidate': 0.979, 'hermes/production': 0.375, 'hermes/candidate': 0.979}}


both agents now run v2


In [10]:
QUESTION = "I bought Pro three weeks ago and barely used it. Can I get a partial refund?"
for runtime, agent in agents.items():
    print(f"[{runtime} · v{live.version}] {agent.ask(QUESTION, tags={'runtime': runtime, 'topic': 'billing', 'prompt_version': str(live.version)})}\n")

[openclaw · v2] No. Refunds are only available within 14 days of purchase, and prorated refunds aren't issued after that window. Since you bought Pro about three weeks ago, your purchase is outside the refund window, so I can't offer a partial refund.



[hermes · v2] No, partial refunds aren't available. Refunds are only available within 14 days of purchase, and Acme Cloud doesn't issue prorated refunds after that window. Since you bought Pro three weeks ago, you're past the 14-day window.



## 5. Rollback is the same move, backwards

If production traffic disagrees with the gate, point the alias back. The bad version stays in the Registry with
its traces linked, for the next analysis.

In [11]:
def rollback_to(version):
    mlflow.genai.set_prompt_alias(PROMPT, alias="production", version=version)
    prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT}@production")
    for agent in agents.values():
        agent.set_agents_md(prompt.template)
    return prompt.version


if passed:
    print("rolled back to", rollback_to(production.version))
    print("and forward again to", rollback_to(candidate.version))

rolled back to 1


and forward again to 2


## Clean up

The gateway is a background process. Stop it here, or leave it running for another pass through the series:
`Gateway.start()` reuses it.

In [12]:
agents["openclaw"].close()
gw.stop()
print("gateway stopped:", not gw.listening())

gateway stopped: True


## Where this goes next

This notebook is Level 2–3 of the tutorial: a model writes the candidate, and code decides whether it ships. The
`levels/` scripts take the same loop further:
* MLflow's issue discovery finds the themes;
* GEPA optimizes the prompt;
* skills (OpenClaw's Skill Workshop) are a second update target;
* the whole loop runs on a schedule, on production traces (Level 5).